In [ ]:
!pip install pycaret==2.0
!pip install shap

     |████████████████████████████████| 256kB 8.6MB/s 
     |████████████████████████████████| 1.2MB 18.8MB/s 
     |████████████████████████████████| 61kB 6.5MB/s 
     |████████████████████████████████| 6.8MB 51.3MB/s 
     |████████████████████████████████| 266kB 54.4MB/s 
     |████████████████████████████████| 102kB 9.0MB/s 
     |████████████████████████████████| 1.6MB 55.4MB/s 
     |████████████████████████████████| 266kB 40.7MB/s 
     |████████████████████████████████| 12.4MB 54.5MB/s 
     |████████████████████████████████| 65.9MB 59kB/s 
     |████████████████████████████████| 235kB 56.0MB/s 
     |████████████████████████████████| 2.1MB 47.3MB/s 
     |████████████████████████████████| 552kB 47.8MB/s 
     |████████████████████████████████| 61kB 5.8MB/s 
     |████████████████████████████████| 71kB 6.9MB/s 
     |████████████████████████████████| 3.1MB 46.4MB/s 
     |████████████████████████████████| 71kB 7.1MB/s 
     |████████████████████████████████| 604kB 36.1MB/s 
  

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive, files
drive.mount('/content/gdrive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly&response_type=code

Enter your authorization code:
··········
Mounted at /content/gdrive


In [ ]:
# Load the data
obs_data_vis = np.load('/content/gdrive/My Drive/Colab Notebooks/Visobs.npy', allow_pickle=True)
model_data_vis = np.load('/content/gdrive/My Drive/Colab Notebooks/Vismodel.npy', allow_pickle=True)

obs_data_T = np.load('/content/gdrive/My Drive/Colab Notebooks/Tobs.npy', allow_pickle=True)
model_data_T = np.load('/content/gdrive/My Drive/Colab Notebooks/Tmodel.npy', allow_pickle=True)

obs_data_Td = np.load('/content/gdrive/My Drive/Colab Notebooks/Tdobs.npy', allow_pickle=True)
model_data_Td = np.load('/content/gdrive/My Drive/Colab Notebooks/Tdmodel.npy', allow_pickle=True)

In [ ]:
# Max model visibility data is 9999 so truncate model vis to that
model_data_vis[model_data_vis > np.max(obs_data_vis)] = np.max(obs_data_vis) 

In [ ]:
# Bin the data
bins = [0, 150, 350, 600, 800, 1500, 3000, 5000, 10000]
obs_data_vis = pd.cut(obs_data_vis, bins, labels=[0,1,2,3,4,5,6,7])
model_binned = np.zeros(model_data_vis.shape)
for i in range(4):
  model_binned[i,:] = pd.cut(model_data_vis[i,:], bins, labels=[0,1,2,3,4,5,6,7])
model_data_vis = model_binned

In [ ]:
# Normalize the Td/T data
obs_data_T = (obs_data_T-np.mean(obs_data_T))/np.std(obs_data_T)
obs_data_Td = (obs_data_Td-np.mean(obs_data_Td))/np.std(obs_data_Td)

model_data_T = (model_data_T-np.mean(model_data_T))/np.std(model_data_T)
model_data_Td = (model_data_Td-np.mean(model_data_Td))/np.std(model_data_Td)

In [ ]:
# Use the past_history nr of data to predict future_target nr of data
past_history = 6
future_target = 3

In [ ]:
# Add obs data to model data
dataset = np.concatenate((model_data_vis, np.reshape(obs_data_vis,(1,-1))), axis=0)
dataset_vis = np.swapaxes(dataset,0,1)

dataset = np.concatenate((model_data_T, np.reshape(obs_data_T,(1,-1))), axis=0)
dataset_T = np.swapaxes(dataset,0,1)

dataset = np.concatenate((model_data_Td, np.reshape(obs_data_Td,(1,-1))), axis=0)
dataset_Td = np.swapaxes(dataset,0,1)

dataset_vis.shape

(27001, 5)

In [ ]:
def multivariate_data(dataset, target, start_index, end_index, history_size,
                      target_size, step, single_step=False):
  data = []
  labels = []

  start_index = start_index + history_size
  if end_index is None:
    end_index = len(dataset) - target_size

  for i in range(start_index, end_index):
    #indices = range(i-history_size, i, step)
    indices = range(i-history_size, i+target_size, step)
    data.append(dataset[indices])

    if single_step:
      labels.append(target[i+target_size])
    else:
      labels.append(target[i:i+target_size])

  return np.array(data), np.array(labels)

In [ ]:
STEP = 1

# This is the visibility (target) dataset
x_train_vis, y_train_vis = multivariate_data(dataset_vis, dataset_vis[:,4], 0,
                                                 None, past_history,
                                                 future_target, STEP)
# These are the auxiliary (T, Td) datasets
x_train_T, _ = multivariate_data(dataset_T, dataset_T[:,4], 0,
                                                 None, past_history,
                                                 future_target, STEP)

x_train_Td, _ = multivariate_data(dataset_Td, dataset_Td[:,4], 0,
                                                 None, past_history,
                                                 future_target, STEP)

In [ ]:
# Using T-Td as feature
x_train_TTd = x_train_T-x_train_Td

In [ ]:
# Keep only 10% of cases where there has been a constant (max) visibility
idx = np.sum(y_train_vis, axis=1) != np.max(np.sum(y_train_vis, axis=1))
idx_2 = np.sum(y_train_vis, axis=1) == np.max(np.sum(y_train_vis, axis=1))

x_train_vis_1 = x_train_vis[idx,];  x_train_vis_2 = x_train_vis[idx_2,]
n_c = np.int(0.1*len(x_train_vis_2))

y_train_vis_1 = y_train_vis[idx,]; y_train_vis_2 = y_train_vis[idx_2,]
x_train_TTd_1 = x_train_TTd[idx,]; x_train_TTd_2 = x_train_TTd[idx_2,]
x_train_T_1 = x_train_T[idx,]; x_train_T_2 = x_train_T[idx_2,]

x_train_vis = np.concatenate((x_train_vis_1, x_train_vis_2[:n_c]))
x_train_TTd = np.concatenate((x_train_TTd_1, x_train_TTd_2[:n_c]))
x_train_T = np.concatenate((x_train_T_1, x_train_T_2[:n_c]))
y_train_vis = np.concatenate((y_train_vis_1, y_train_vis_2[:n_c]))

In [ ]:
# Set the future obs to the average of the model data
x_train_vis[:,past_history:,4] = np.round(np.average(x_train_vis[:,past_history:,0:4], axis=2))
x_train_T[:,past_history:,4] = np.round(np.average(x_train_T[:,past_history:,0:4], axis=2))
x_train_Td[:,past_history:,4] = np.round(np.average(x_train_Td[:,past_history:,0:4], axis=2))

In [ ]:
# check version
from pycaret.utils import version
version()

2.0


In [ ]:
print(y_train_vis.shape)

(4879, 3)


In [ ]:
keys_y = ['y_vis']
keys_vis_obs_x = ['x_vis_obs_step_' + str(k) for k in range(past_history+future_target)]
keys_vis_model_x = ['x_vis_model_step_' + str(k) for k in range(past_history+future_target)]
keys_T_obs_x = ['x_T_obs_step_' + str(k) for k in range(past_history+future_target)]
keys_T_model_x = ['x_T_model_step_' + str(k) for k in range(past_history+future_target)]
keys_TTd_obs_x = ['x_TTd_obs_step_' + str(k) for k in range(past_history+future_target)]
keys_TTd_model_x = ['x_TTd_model_step_' + str(k) for k in range(past_history+future_target)]
keys = keys_y + keys_vis_obs_x + keys_vis_model_x + keys_T_obs_x + \
       keys_T_model_x + keys_TTd_obs_x + keys_TTd_model_x

In [ ]:
# Change in y_train_vis the second index to 0, 1, 2, ..., future_step-1
vals = np.concatenate((np.expand_dims(y_train_vis[:,0],1), x_train_vis[:,:,4], 
                       x_train_vis[:,:,0:4].mean(axis=2), 
                       x_train_T[:,:,4],
                       x_train_T[:,:,0:4].mean(axis=2),
                       x_train_TTd[:,:,4],
                       x_train_TTd[:,:,0:4].mean(axis=2)), axis = 1)

In [ ]:
data = {}
j = 0
for i in keys:
  data[i] = pd.Series(vals[:,j])
  j += 1
data = pd.DataFrame(data)

In [ ]:
data = data[data.y_vis != 0]

In [ ]:
from pycaret.classification import *
clf = setup(data, target='y_vis', session_id=123, log_experiment=True, train_size=0.9,
            experiment_name='step_0')

IntProgress(value=0, description='Processing: ', max=13)

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,18:17:25
Status,. . . . . . . . . . . . . . . . . .,Preparing Data for Modeling
ETC,. . . . . . . . . . . . . . . . . .,Calculating ETC


Text(value="Following data types have been inferred automatically, if they are correct press enter to continue…

,Data Type
y_vis,Label
x_vis_obs_step_0,Numeric
x_vis_obs_step_1,Numeric
x_vis_obs_step_2,Numeric
x_vis_obs_step_3,Numeric
x_vis_obs_step_4,Numeric
x_vis_obs_step_5,Numeric
x_vis_obs_step_6,Numeric
x_vis_obs_step_7,Numeric
x_vis_obs_step_8,Numeric


In [ ]:
best_model = compare_models(blacklist = ['catboost'])

In [ ]:
xgb = create_model('xgboost')

In [ ]:
# # Create ensemble models
# bagged_xgb = ensemble_model(xgb)

In [ ]:
# Plot model
plot_model(xgb)

In [ ]:
# # Predict model
# predictions = predict_model(xgb)

In [ ]:
# Predict model
predictions = predict_model(xgb)

In [ ]:
# AutoML
# best = automl(optimize='Accuracy')